# 프로젝트 개요




## Kaggle 제출 가이드 (요약)

1. Kaggle 대회 페이지에서 **Data** 탭의 `train.csv`, `test.csv`, `sample_submission.csv` 구조를 확인합니다.
2. 이 노트북을 실행해 `submission.csv`를 생성합니다.
3. Kaggle 대회 페이지 → **Submit Predictions** → `submission.csv` 업로드
4. 제출 파일 형식은 `sample_submission.csv`와 동일해야 합니다.
   - 컬럼: `id`, `health_condition`
   - `id` 순서와 개수는 `test.csv`와 같아야 합니다.
5. Public Leaderboard 점수는 제출 후 확인합니다.

**왜 Accuracy 대신 Balanced Accuracy를 보나?**
- 현재 데이터는 `at-risk` 클래스 비율이 매우 높아 일반 Accuracy만 보면 다수 클래스에 치우친 모델도 높게 나올 수 있습니다.
- `balanced_accuracy_score`는 **각 클래스 recall의 평균**이라 불균형 다중분류에서 더 해석하기 좋습니다.
- scikit-learn 문서도 불균형 분류에서 balanced accuracy 사용을 안내합니다.


## 프로젝트 가이드라인
- 7월 15일 수요일
  + 3교시까지 개인별 or 팀별 / 개인 점수 제출
  + 4교시까지 발표자료 제출 (PPT)
- 발표자료에 반드시 들어갈 내용
  + 발표자료 간지포함(30페이지 내)
    - 자세한 부분은 모두 부록으로 치환
  + 캐글 대회 소개 및 데이터 정의서 정리
  + 프로젝트 정의 (SCQA), 왜 머신러닝 프로젝트가 필요하며, 과정 도식화
  + 프로젝트 수행
    - 탐색적 자료분석 (시각화 및 통계 검정)
    - 머신러닝 모델 비교표 (최소 10개 모델 수행)
      + 시간 측정 필수 (시간 대비 평가지표 비교하는 시각화 필수)
    - 머신러닝 아키텍쳐 (Pipeline 포함)
    - 최종 제출 점수 (ID와 점수 모두 표시되도록 캡쳐)
  + 액션플랜
    - 개발된 모델을 어떤 형태로 프로그램화 할 것인지 개념화
  + 한계점 및 향후 계획
- 프로젝트 조언
  + 절대 바이브코딩 (코드 입력하면서 하기엔 시간이 부족합니다!)
  + 전체 데이터 활용하여 모델 비교표 만들면 시간내에 못합니다. 샘플링 적용 필수
  + 프로젝트의 전체 흐름을 이해하는 것이 이번 미니 프로젝트의 목적
  + 7월 중순 이후, 의미있는 개인 프로젝트로 잘 정리할 것


# 라이브러리 불러오기

In [ ]:
# 필요한 라이브러리 import
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# 데이터 불러오기

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# 1. 데이터 로드
# ============================================================
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/kaggle_student_classification/'

train_df = pd.read_csv(DATA_PATH + 'train.csv')
test_df = pd.read_csv(DATA_PATH + 'test.csv')
sample_submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
print('sample_submission shape:', sample_submission.shape)
print()
print('target classes:')
print(train_df['health_condition'].value_counts())
print()
print('target ratio:')
print(train_df['health_condition'].value_counts(normalize=True).round(4))
print()
train_df.head()


train shape: (690088, 15)
test shape : (295753, 14)
sample_submission shape: (295753, 2)

target classes:
health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64

target ratio:
health_condition
at-risk      0.8587
unhealthy    0.0836
fit          0.0577
Name: proportion, dtype: float64



,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


# 변수 설명

Kaggle 데이터셋 설명을 바탕으로 각 변수를 아주 간단히 정리하면 아래와 같습니다.

| 변수명 | 한글 설명 |
|---|---|
| `sleep_duration` | 하루 평균 수면 시간 |
| `heart_rate` | 안정 시 심박수 또는 평균 심박수 |
| `bmi` | 체질량지수 |
| `calorie_expenditure` | 하루 칼로리 소모량 |
| `step_count` | 하루 걸음 수 |
| `exercise_duration` | 운동 시간 |
| `water_intake` | 하루 물 섭취량 |
| `diet_type` | 식단 유형 (`veg`, `non-veg`, `balanced` 등) |
| `stress_level` | 스트레스 수준 (`low`, `medium`, `high`) |
| `sleep_quality` | 수면의 질 (`poor`, `average`, `good`) |
| `physical_activity_level` | 신체 활동 수준 (`sedentary`, `moderate`, `active`) |
| `smoking_alcohol` | 흡연/음주 여부 또는 빈도 |
| `gender` | 성별 |
| `health_condition` | 예측 대상 건강 상태 (`fit`, `at-risk`, `unhealthy`) |

- 일반적인 건강 위험도 분류 방식을 기준으로 정리해 드릴게요.
- 타겟 클래스 (건강 상태 등급) 의미

| 클래스 | 의미 | 일반적으로 대응되는 특성 |
|---|---|---|
| **Fit** | 균형 잡히고 건강한 상태 | 규칙적인 운동, 충분한 수면(7~9시간), 균형 잡힌 식습관, 낮은 스트레스 수준, 정상 체중(BMI) |
| **At-Risk** | 중간 수준의 건강 우려가 있는 상태 | 불규칙한 생활 패턴(수면 부족·과다, 운동 부족), 스트레스 증가, 식습관 일부 불균형 — 아직 심각하지는 않지만 개선이 필요한 경계 상태 |
| **Unhealthy** | 높은 수준의 건강 위험이 있는 상태 | 만성적 수면 부족, 신체 활동 거의 없음, 흡연/음주/패스트푸드 등 위험 행동 빈번, 높은 스트레스·불안 지표, 비정상 체중 등 복합적 위험 요인 |


In [ ]:
# ============================================================
# 2. feature / target 분리
# ============================================================
target_col = 'health_condition'
id_col = 'id'

feature_cols = [
    'sleep_duration',
    'heart_rate',
    'bmi',
    'calorie_expenditure',
    'step_count',
    'exercise_duration',
    'water_intake',
    'diet_type',
    'stress_level',
    'sleep_quality',
    'physical_activity_level',
    'smoking_alcohol',
    'gender',
]

X = train_df.loc[:, feature_cols].copy()
y = train_df.loc[:, target_col].copy()
X_test_kaggle = test_df.loc[:, feature_cols].copy()
test_ids = test_df.loc[:, id_col].copy()

print('X shape:', X.shape)
print('y shape:', y.shape)
print('X_test_kaggle shape:', X_test_kaggle.shape)
print()
print('결측치 개수 (train):')
print(X.isnull().sum())


X shape: (690088, 13)
y shape: (690088,)
X_test_kaggle shape: (295753, 13)

결측치 개수 (train):
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64
